# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nglfrsarthak/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane: growth / momentum on Google — same March decision moment as W03 (features Mar 1–15, outcome Mar 16–31). This notebook checks two flag-linked signals, encodes one hand-written rule, writes the ranked queue, and reviews its own top 10.

> Skills loaded: `building-baselines` + `flyrank/flyrank-data` (see `skills/README.md`). Lane stays as declared in W03.

In [ ]:
%pip -q install duckdb pandas scikit-learn

In [ ]:
import os, getpass, pathlib, json

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
print('month:', REL + '/fact_content_daily_performance/month=2026-03/')

## 1. My rule and its reason codes — and the two signals it leans on

**Rule in plain words (what the session built live):**

> A page is worth a refresh if it was visible in early March, it is stale (not updated in a long time), and its Google position is already slipping past page one.

**One rule, one reason code, one action.** Score `0..2` = how many of the two risk conditions fire, gated by visibility. Reason code `stale_visible_slipping` tags every scored item. Action `refresh_candidate` is the human-readable label.

Two signals checked below — both flag-linked:

1. **Staleness behind the refresh flags** — does older `days_since_last_update` line up with the W03 proxy decline?
2. **CTR-vs-position behind the CTR-fix logic** — does lower (worse) position line up with lower CTR at fixed volume? (FlyRank's `needs_ctr_fix` leans on this.)

In [ ]:
# Build the March per-content frame ONCE — GSC-side only, decision moment 2026-03-15.
# Join staleness from dim_content (one row per content) with ANY_VALUE — never SUM a per-item column.
frame = con.sql(f"""
    WITH march AS (
        SELECT content_hash_id,
               SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first15,
               SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_first15,
               AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS avg_pos_first15,
               SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_seen_first15,
               SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last16
        FROM {MONTH}
        GROUP BY 1
        HAVING imp_first15 >= 100
    )
    SELECT m.*,
           ANY_VALUE(d.days_since_last_update) AS days_since_last_update,
           ANY_VALUE(d.content_age_days) AS content_age_days
    FROM march m
    LEFT JOIN {DIM_CONTENT} d USING (content_hash_id)
""").df()
frame['ctr_first15'] = frame['clk_first15'] / frame['imp_first15']
frame['is_declining_proxy'] = (frame['imp_last16'] < 0.8 * frame['imp_first15']).astype(int)
print(f"content items with enough history: {len(frame):,}")
print(f"measured proxy decline rate (directional, not a claim): {frame['is_declining_proxy'].mean():.3f}")
# Staleness coverage: per-item column repeats — check how many actually have it.
print(f"rows with staleness present: {frame['days_since_last_update'].notna().sum():,} / {len(frame):,}")
frame.head(3)

In [ ]:
# Signal 1 — STALENESS (flag-linked: refresh flags). Bucket n printed.
# Bins chosen to mirror the dictionary tiers; verdict is directional, not a model claim.
import pandas as pd
tmp = frame.dropna(subset=['days_since_last_update']).copy()
tmp['stale_bucket'] = pd.cut(tmp['days_since_last_update'], bins=[-1, 30, 90, 180, 1e9], labels=['0-30','31-90','91-180','181+'])
s1 = tmp.groupby('stale_bucket', observed=True).agg(n=('is_declining_proxy','size'), decline_rate=('is_declining_proxy','mean')).reset_index()
print(s1.to_string(index=False))
# One-word verdict (read the table — a negative is a win): check if 181+ > 0-30 directionally.
verdict_s1 = 'CONFIRMED' if len(s1) >= 2 and float(s1.iloc[-1]['decline_rate']) > float(s1.iloc[0]['decline_rate']) else 'MIXED'
print('Verdict S1 (staleness -> decline):', verdict_s1)

In [ ]:
# Signal 2 — CTR-vs-POSITION (flag-linked: CTR-fix / quick-win). Bucket n printed.
# Guard: require imp_first15 >= 500 and 0 < pos <= 20 so a few impressions don't swing CTR (dictionary warning).
tmp2 = frame[(frame['avg_pos_first15'].notna()) & (frame['avg_pos_first15'] > 0) & (frame['imp_first15'] >= 500)].copy()
tmp2['pos_bucket'] = pd.cut(tmp2['avg_pos_first15'], bins=[0, 3, 10, 20, 1e9], labels=['top_3','page_1','striking','deep'])
s2 = tmp2.groupby('pos_bucket', observed=True).agg(n=('ctr_first15','size'), median_ctr=('ctr_first15','median'), median_imp=('imp_first15','median')).reset_index()
print(s2.to_string(index=False))
verdict_s2 = 'CONFIRMED' if len(s2) >= 2 and float(s2.iloc[0]['median_ctr']) > float(s2.iloc[-1]['median_ctr']) else 'MIXED'
print('Verdict S2 (worse position -> lower CTR):', verdict_s2)

Both verdicts are directional decision-support, not causal claims. A MIXED/OPPOSITE is an honest win — it just saved the rule from leaning on noise.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [ ]:
# ONE rule, transparent score, no fitted weights, no future window, no label-derived input.
import numpy as np, pathlib, json
df = frame.copy()
visible = (df['imp_first15'] >= 100).astype(int)  # same floor as W03
stale   = (df['days_since_last_update'] >= 90).fillna(False).astype(int)  # 90d = freshness tier boundary
slipping = ((df['avg_pos_first15'] > 10) & (df['avg_pos_first15'] <= 20)).fillna(False).astype(int)  # page-1 edge / striking

df['score'] = visible * (stale + slipping)  # 0..2, readable on purpose
df['reason_code'] = np.where(df['score'] > 0, 'stale_visible_slipping', 'not_flagged')
df['action'] = np.where(df['score'] > 0, 'refresh_candidate', 'no_action')
df['label_proxy'] = df['is_declining_proxy']  # kept for honest precision@K only — never a feature

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean()) if k > 0 else float('nan')

base_rate = float(df['label_proxy'].mean())
for k in [10, 20, 50, 100]:
    print(f"precision@{k}: {precision_at_k(df['score'], df['label_proxy'], k):.3f}  (base_rate {base_rate:.3f})")
print(f"scored (score>0): {(df['score']>0).sum():,} / {len(df):,}")
df.head(3)

In [ ]:
# Write the CSV FROM the notebook — this file is gitignored by design; the notebook regenerates it.
out = pathlib.Path('work/outputs/baseline_action_score.csv')
out.parent.mkdir(parents=True, exist_ok=True)
cols = ['content_hash_id','score','reason_code','action','imp_first15','avg_pos_first15','days_since_last_update','ctr_first15','label_proxy']
ranked = df.sort_values(['score','imp_first15'], ascending=[False, False])
ranked[cols].to_csv(out, index=False)
print(f'wrote {out} — {len(ranked):,} rows, {ranked[cols].head(1).to_dict(orient="records")[0]}')
# Keep a commit-friendly receipt (JSON is not gitignored).
metrics = {'base_rate': float(df['label_proxy'].mean()), 'scored': int((df['score']>0).sum()), 'total': int(len(df)),
           'precision@10': precision_at_k(df['score'], df['label_proxy'], 10),
           'precision@20': precision_at_k(df['score'], df['label_proxy'], 20),
           'precision@50': precision_at_k(df['score'], df['label_proxy'], 50)}
pathlib.Path('work/outputs/baseline_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))

## 3. Top-10 review

*For each of the top 10: action, reason code, confidence note, and what would make it wrong.*

Table is generated; the “what would make it wrong” line is the skeptic's eye the brief asks for.

In [ ]:
top10 = ranked.head(10).reset_index(drop=True)
for i, r in top10.iterrows():
    print(f"{i+1:2d}. action={r['action']}  reason={r['reason_code']}  score={int(r['score'])}  "
          f"imp15={int(r['imp_first15'])}  pos={r['avg_pos_first15']:.1f}  stale={int(r['days_since_last_update']) if pd.notna(r['days_since_last_update']) else 'NA'}  ctr={r['ctr_first15']:.3f}%")
    print(f"    confidence: {'high' if r['score']==2 and r['imp_first15']>=500 else 'medium' if r['score']==2 else 'low-medium'}; "
          f"would be wrong if: position is noisy on low volume, staleness is not causal here, or the March half-split is seasonal noise, not real decay.")


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# Weak picks: surface at least one that looks shaky even if it scored high.
weak = ranked[(ranked['score']==1) & (ranked['imp_first15'] < 200)].head(3)
print('weak examples (score 1 but low volume — noisy CTR/pos):')
print(weak[['score','imp_first15','avg_pos_first15','ctr_first15','days_since_last_update']].to_string(index=False))
print()
# Leakage check: every input was Mar 1–15 or a per-content dim; label/output window was strictly Mar 16–31.
print('leakage check:')
print('- No fact_content_query_90d columns used (its 90d window overlaps the outcome).')
print('- No product flags (health_score, priority_score, action_type) — not in data, never referenced.')
print('- No label-derived inputs (is_declining_proxy / imp_last16 / trend_pct) in the score — only in evaluation.')
print('- No future dates: all features aggregate report_date <= 2026-03-15.')
print('- IDs (content_hash_id) used only for grouping/joining, never as a feature.')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — run it in Colab, then tick
- [x] No client names, URLs, or private queries anywhere (hash IDs only, never printed verbatim)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- CSV is gitignored by design; `work/outputs/baseline_metrics.json` is the committed receipt.